[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IbHansen/wb-debt-simulation/blob/main/optimization/currency.ipynb)

In [1]:
if 'google.colab' in str(get_ipython()):
    import os
    os.system('pip -qqq install ModelFlowIb')
    os.system('curl -L -o exchangerates_get.py https://raw.githubusercontent.com/IbHansen/wb-debt-simulation/main/optimization/exchangerates_get.py')

In [2]:
import pandas as pd
import numpy as np

import exchangerates_get as er

# Flip to True to force a fresh ECB download and overwrite the cache.
REFRESH = False


In [3]:
fx_eur = er.ecb_fx_eur(
    currencies=["USD", "GBP", "JPY", "CHF","EUR","ZAR"],
    start="2010-01-01",
    freq='Q',
    refresh=REFRESH,
)
#fx_eur    

CURRENCY,EUR_CHF,EUR_GBP,EUR_JPY,EUR_USD,EUR_ZAR,EUR_EUR
TIME_PERIOD,,,,,,
2010Q1,1.4276,0.88980,125.93,1.3479,9.8922,1.0
2010Q2,1.3283,0.81745,108.79,1.2271,9.3808,1.0
2010Q3,1.3287,0.85995,113.68,1.3648,9.5438,1.0
2010Q4,1.2504,0.86075,108.65,1.3362,8.8625,1.0
2011Q1,1.3005,0.88370,117.61,1.4207,9.6507,1.0
...,...,...,...,...,...,...
2025Q2,0.9347,0.85550,169.17,1.1720,20.8411,1.0
2025Q3,0.9364,0.87340,173.76,1.1741,20.2820,1.0
2025Q4,0.9314,0.87260,184.09,1.1750,19.4439,1.0


In [4]:
# Step 2: express everything in base currency 
fx_ccy = er.convert_base_currency(fx_eur, base="zar")

fx_returns = er.get_fx_returns(fx_ccy)
# Step 3: yearly covariance matrices
fx_cov = er.get_fx_covariance(fx_returns)

In [7]:
cov_df = fx_cov.rename(
    index=lambda x: x.split('_')[1],
    columns=lambda x: x.split('_')[1],
)
names = cov_df.index

assumptions = pd.DataFrame(
    {
        'interest_rate':         [0.010, 0.020, 0.023, 0.013, 0.034],
        'expected_appreciation': [0.000, 0.000, 0.000, 0.000, 0.000],
        'min_share':             [0.0, 0.0, 0.0, 0.0, 0.0],
        'max_share':             [1.0, 1.0, 1.0, 1.0, 1.0],
        'current_share':         [0.2, 0.2, 0.2, 0.2, 0.2],
    },
    index=names,
)



## Interactive inputs — `DebtFrontierInputs`

The cells below wrap the same covariance matrix and assumption frame in a `DebtFrontierInputs` dataclass and render an editable grid (currencies × input fields). Edit any cell, then press **Run frontier** to re-solve and re-plot. The dataclass's `assumptions` attribute always reflects the latest edits, so you can also drive `inputs.plot()` from code between clicks.

In [8]:
inputs = er.DebtFrontierInputs(cov_df=cov_df, assumptions=assumptions)
inputs.assumptions

,interest_rate,expected_appreciation,min_share,max_share,current_share
CHF,0.010,0.0,0.0,1.0,0.2
GBP,0.020,0.0,0.0,1.0,0.2
JPY,0.023,0.0,0.0,1.0,0.2
USD,0.013,0.0,0.0,1.0,0.2
EUR,0.034,0.0,0.0,1.0,0.2


In [9]:
inputs.widget()